In [1]:
%load_ext autoreload
%autoreload 2
# 1. 导入所有依赖库
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "2" 
import torch
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm
import numpy as np
import random

# 导入我们的核心模块
from models.apm_former import APM_Former_ImageOnly
from utils.dataset import get_adni_dataloaders
# 修正导入：只导入config里存在的变量
from utils.config import TRAIN_CSV, VAL_CSV, BATCH_SIZE, NUM_WORKERS, TRAIN_IMG_SIZE
from utils.logging_utils import get_logger
from torch.optim.lr_scheduler import SequentialLR, LinearLR, CosineAnnealingLR

import torch.nn.functional as F
from sklearn.metrics import confusion_matrix, accuracy_score, recall_score, precision_score, f1_score, roc_auc_score


# 日志
logger = get_logger("Train")

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    # torch.cuda.manual_seed_all(seed)  # 如果用多卡
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False



# seed 42 
# seed 3407 ACC: 81.20% | SEN: 61.54% | SPE: 91.03% | F1: 68.57% | AUC: 86.26%    ACC: 80.34% | SEN: 56.41% | SPE: 92.31% | F1: 65.67% | AUC: 83.86%
# seed 1234 
# seed 98 
# seed 20  
# seed 1024


In [2]:
# 2. 超参数配置
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
LEARNING_RATE = 3e-4
UNFREEZE_EPOCH = 5  # 设定前 5 个 epoch 为阶段一
WEIGHT_DECAY = 1e-3
NUM_EPOCHS = 50
GRADIENT_ACCUMULATION_STEPS = 1
NUM_CLASSES = 2
FEATURE_SIZE = 24
GUIDE_CHANNELS = 18
SAVE_PATH = "checkpoints/best_adcn3407_model2.pth"
PRETRAINED_SWIN_PATH = "checkpoints/model_swinvit.pt"
SEED = 3407

set_seed(SEED)

logger.info("=" * 50)
logger.info("🌟 本次实验配置档案 (Experiment Config) 🌟")
logger.info("=" * 50)
logger.info(f"▶ 随机种子 (SEED): {SEED}")
logger.info(f"▶ 峰值学习率 (LR): {LEARNING_RATE}")
logger.info(f"▶ 预热轮数 (Warmup Epochs): {UNFREEZE_EPOCH}")
logger.info(f"▶ 批次大小 (Batch Size): {BATCH_SIZE}")
logger.info(f"▶ 优化器: AdamW (Weight Decay: {WEIGHT_DECAY})")
logger.info(f"▶ 架构说明: 启用浅层特征多尺度融合 (shallow_downsample)")
logger.info(f"▶ 架构说明: 启用 m_k 调制掩码")
logger.info("=" * 50)

logger.info(f"训练设备: {DEVICE}")
logger.info(f"批次大小: {BATCH_SIZE}")
logger.info(f"总轮数: {NUM_EPOCHS}")

2026-05-22 10:30:04,077 - Train - INFO - ==================================================
2026-05-22 10:30:04,078 - Train - INFO - 🌟 本次实验配置档案 (Experiment Config) 🌟
2026-05-22 10:30:04,079 - Train - INFO - ==================================================
2026-05-22 10:30:04,081 - Train - INFO - ▶ 随机种子 (SEED): 3407
2026-05-22 10:30:04,081 - Train - INFO - ▶ 峰值学习率 (LR): 0.0003
2026-05-22 10:30:04,083 - Train - INFO - ▶ 预热轮数 (Warmup Epochs): 5
2026-05-22 10:30:04,083 - Train - INFO - ▶ 批次大小 (Batch Size): 2
2026-05-22 10:30:04,084 - Train - INFO - ▶ 优化器: AdamW (Weight Decay: 0.001)
2026-05-22 10:30:04,085 - Train - INFO - ▶ 架构说明: 启用浅层特征多尺度融合 (shallow_downsample)
2026-05-22 10:30:04,086 - Train - INFO - ▶ 架构说明: 启用 m_k 调制掩码
2026-05-22 10:30:04,087 - Train - INFO - ==================================================
2026-05-22 10:30:04,088 - Train - INFO - 训练设备: cuda
2026-05-22 10:30:04,089 - Train - INFO - 批次大小: 2
2026-05-22 10:30:04,090 - Train - INFO - 总轮数: 50


In [3]:
# 3. 加载训练集 + 验证集 DataLoader
logger.info("正在加载数据集...")
train_loader, val_loader = get_adni_dataloaders(
    train_csv=TRAIN_CSV,
    val_csv=VAL_CSV,
    batch_size=BATCH_SIZE,
    target_size=TRAIN_IMG_SIZE,
    num_workers=NUM_WORKERS
)

2026-05-22 10:30:04,135 - Train - INFO - 正在加载数据集...
2026-05-22 10:30:04,171 - utils.dataset - INFO - ✅ 成功加载 612 个样本
2026-05-22 10:30:04,177 - utils.dataset - INFO - ✅ 成功加载 77 个样本
2026-05-22 10:30:04,179 - utils.dataset - INFO - ✅ DataLoader 构建完成 (已启用 WeightedRandomSampler)


In [4]:
# 4. 初始化模型（完整版！）
logger.info("初始化 APM-Former 模型...")
model = APM_Former_ImageOnly(
    img_size=TRAIN_IMG_SIZE,
    in_channels=1,
    num_classes=NUM_CLASSES,
    feature_size=FEATURE_SIZE,
    guide_channels=GUIDE_CHANNELS,
    pretrained_swin_path=PRETRAINED_SWIN_PATH # 传入刚刚下载的权重
).to(DEVICE)

# 打印模型参数量
total_params = sum(p.numel() for p in model.parameters())
logger.info(f"模型总参数量: {total_params / 1e6:.2f} M")

2026-05-22 10:30:04,221 - Train - INFO - 初始化 APM-Former 模型...


/home/ubuntu/Code/APM_Former_Project/models/apm_former.py:25: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(pretrained_swin_path, map_location='cpu')

加载SwinUNETR预训练权重：checkpoints/model_swinvit.pt


2026-05-22 10:30:05,557 - Train - INFO - 模型总参数量: 16.32 M


In [5]:
# 5. 损失函数 + 两阶段优化器设置
from monai.losses import FocalLoss
from torch.optim.lr_scheduler import CosineAnnealingLR

# ==========================================
# 损失函数 (保持你原来的完美配置不变)
# ==========================================
weights = torch.tensor([1.0, 1.0]).to(DEVICE) 
criterion = FocalLoss(weight=weights, gamma=2.0, to_onehot_y=True).to(DEVICE)

# ==========================================
# 两阶段训练：阶段一 (冻结主干)
# ==========================================

# 1. 冻结 Swin 主干网络
logger.info("🔒 阶段一：冻结 Swin 主干网络，仅训练新增模块...")
for name, param in model.named_parameters():
    if "swin_backbone" in name:
        param.requires_grad = False
    else:
        param.requires_grad = True

# 2. 定义优化器：极其重要！这里加了 filter，只把没冻结的参数交给优化器
optimizer = optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()), 
    lr=3e-4, 
    weight_decay=WEIGHT_DECAY
)

# 3. 阶段一的调度器：不需要预热，直接用 5 个 epoch 的余弦退火
scheduler = CosineAnnealingLR(optimizer, T_max=UNFREEZE_EPOCH)

2026-05-22 10:30:05,606 - Train - INFO - 🔒 阶段一：冻结 Swin 主干网络，仅训练新增模块...


In [6]:
# 6. 训练/验证函数
def train_epoch(model, loader, criterion, optimizer, device, accum_steps):
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0

    optimizer.zero_grad()
    for idx, (images, labels) in enumerate(tqdm(loader, desc="训练中")):
        images = images.to(device)
        labels = labels.to(device)

        logits, _, _ = model(images)
        loss = criterion(logits, labels.unsqueeze(1))
        # loss = criterion(logits, labels)
        loss = loss / accum_steps

        loss.backward()
        # 👇 就在 optimizer.step() 之前，加入这行“梯度裁剪”代码！
        # max_norm=1.0 或 2.0 是最常用的安全值
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        if (idx + 1) % accum_steps == 0:
            optimizer.step()
            optimizer.zero_grad()

        total_loss += loss.item() * accum_steps
        preds = torch.argmax(logits, dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    avg_loss = total_loss / len(loader)
    acc = 100 * correct / total
    return avg_loss, acc

from sklearn.metrics import confusion_matrix

@torch.no_grad()
def val_epoch(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    # correct = 0
    # total = 0
    all_preds = []
    all_labels = []
    all_probs = []  # 🌟 新增：专门用于收集预测为 pMCI 的概率，计算 AUC 必须用

    for images, labels in loader: 
        images = images.to(device)
        labels = labels.to(device)

        logits, _, _ = model(images)
        loss = criterion(logits, labels.unsqueeze(1)) # 如果用的是交叉熵，这里不用 unsqueeze

        total_loss += loss.item()
        
        # 获取预测类别 (0 或 1)
        preds = torch.argmax(logits, dim=1)
        
        # 🌟 获取预测为类别 1 (pMCI) 的概率
        probs = F.softmax(logits, dim=1)[:, 1] 
              

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())

    avg_loss = total_loss / len(loader)
    
    # ==========================
    # 🌟 计算所有医学黄金指标
    # ==========================
    acc = accuracy_score(all_labels, all_preds) * 100
    sen = recall_score(all_labels, all_preds, pos_label=1) * 100  # Sensitivity / Recall
    spe = recall_score(all_labels, all_preds, pos_label=0) * 100  # Specificity (反向召回)
    f1 = f1_score(all_labels, all_preds, pos_label=1) * 100
    
    # 计算 AUC (注意：有些 batch 如果只有一个类别会报错，这里做了容错)
    try:
        auc = roc_auc_score(all_labels, all_probs) * 100
    except ValueError:
        auc = 0.0

    cm = confusion_matrix(all_labels, all_preds, labels=[0, 1])
    
    # 打印超级豪华版日志
    logger.info(f"👉 混淆矩阵: [sMCI正确: {cm[0][0]}, 误判为pMCI: {cm[0][1]}] | [pMCI漏判: {cm[1][0]}, pMCI正确: {cm[1][1]}]")
    logger.info(f"🏆 评估指标: ACC: {acc:.2f}% | SEN: {sen:.2f}% | SPE: {spe:.2f}% | F1: {f1:.2f}% | AUC: {auc:.2f}%")
    
    return avg_loss, acc, auc

In [7]:
# 7. 主训练循环
best_val_acc = 0.0
logger.info("=" * 50)
logger.info("开始训练！")
logger.info("=" * 50)

for epoch in range(NUM_EPOCHS):
    # 👇 ================= 新增：解冻机关 ================= 👇
    if epoch == UNFREEZE_EPOCH:
        logger.info("\n" + "🚀" * 20)
        logger.info("🔓 阶段二触发：解冻 Swin 主干网络，开始全员微调！")
        logger.info("🚀" * 20)
        
        # 将所有参数解冻
        for param in model.parameters():
            param.requires_grad = True
            
        # 重新定义优化器（包含主干网络），并大幅降低学习率来保护老专家！
        # 这里用 1e-4 或 5e-5 进行微调最合适
        optimizer = optim.AdamW(model.parameters(), lr=1e-5, weight_decay=WEIGHT_DECAY)
        
        # 重新定义剩下 epoch 的调度器
        scheduler = CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS - UNFREEZE_EPOCH)
    # 👆 =================================================== 👆
    logger.info(f"\nEpoch [{epoch+1}/{NUM_EPOCHS}]")
    
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, DEVICE, GRADIENT_ACCUMULATION_STEPS)
    val_loss, val_acc, val_auc = val_epoch(model, val_loader, criterion, DEVICE)
    scheduler.step()

    logger.info(f"训练损失: {train_loss:.4f} | 训练准确率: {train_acc:.2f}%")
    logger.info(f"验证损失: {val_loss:.4f} | 验证准确率: {val_acc:.2f}%")
    logger.info(f"当前学习率: {optimizer.param_groups[0]['lr']:.8f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), SAVE_PATH)
        logger.info(f"✅ 最佳模型已保存！最佳验证ACC: {best_val_acc:.2f}%")

logger.info("\n🎉 训练完成！")
logger.info(f"最终最佳验证ACC: {best_val_acc:.2f}%")

2026-05-22 10:30:05,695 - Train - INFO - ==================================================
2026-05-22 10:30:05,697 - Train - INFO - 开始训练！
2026-05-22 10:30:05,698 - Train - INFO - ==================================================
2026-05-22 10:30:05,700 - Train - INFO - 
Epoch [1/50]
训练中: 100%|██████████| 306/306 [01:11<00:00,  4.29it/s]
2026-05-22 10:31:26,919 - Train - INFO - 👉 混淆矩阵: [sMCI正确: 0, 误判为pMCI: 38] | [pMCI漏判: 0, pMCI正确: 39]
2026-05-22 10:31:26,921 - Train - INFO - 🏆 评估指标: ACC: 50.65% | SEN: 100.00% | SPE: 0.00% | F1: 67.24% | AUC: 75.44%
2026-05-22 10:31:26,922 - Train - INFO - 训练损失: 0.1750 | 训练准确率: 51.47%
2026-05-22 10:31:26,923 - Train - INFO - 验证损失: 0.1735 | 验证准确率: 50.65%
2026-05-22 10:31:26,924 - Train - INFO - 当前学习率: 0.00027135
2026-05-22 10:31:28,091 - Train - INFO - ✅ 最佳模型已保存！最佳验证ACC: 50.65%
2026-05-22 10:31:28,093 - Train - INFO - 
Epoch [2/50]
训练中: 100%|██████████| 306/306 [01:12<00:00,  4.25it/s]
2026-05-22 10:32:49,648 - Train - INFO - 👉 混淆矩阵: [sMCI正确: 38, 误判为pM

In [8]:
# import torch
# import torch.nn.functional as F
# import numpy as np
# import matplotlib.pyplot as plt

# # ==========================================
# # 第一步：加载你跑出 74.39% 的最佳模型权重
# # ==========================================
# print("🔄 正在加载最佳模型权重...")

# # 实例化模型 (确保这里的参数和你跑出最好成绩的那次完全一致)
# # 之前确认过 FEATURE_SIZE 是 24
# vis_model = APM_Former_ImageOnly(
#     feature_size=24,  
#     num_classes=2
# ).to(DEVICE)

# # ⚠️ 注意：把这里的 'best_model.pth' 换成你实际保存的权重文件名！
# # 比如可能是 'checkpoint_epoch_xx.pth'
# checkpoint_path = 'checkpoints/best_model.pth'  
# state_dict = torch.load(checkpoint_path, map_location=DEVICE)
# vis_model.load_state_dict(state_dict)

# vis_model.eval() # 切换到评估模式
# print(f"✅ 成功加载权重: {checkpoint_path}")
# print("-" * 50)


# # ==========================================
# # 第二步：定义并运行可视化探针
# # ==========================================
# @torch.no_grad()
# def visualize_pMCI_attention(model, val_loader, device, num_cases_to_show=3):
#     model.eval()
#     found_cases = 0
    
#     # 解决 matplotlib 中文显示问题
#     # plt.rcParams['font.sans-serif'] = ['SimHei'] 
#     # plt.rcParams['axes.unicode_minus'] = False
    
#     print("🕵️‍♂️ 探针已启动，正在验证集中搜寻成功预测的 pMCI 病例...\n")
    
#     for images, labels in val_loader:
#         images = images.to(device)
#         labels = labels.to(device)
        
#         logits, _, spatial_attention = model(images)
#         preds = torch.argmax(logits, dim=1)
        
#         # 寻找 True Positives (真实为pMCI，预测也为pMCI)
#         for i in range(len(labels)):
#             if labels[i].item() == 1 and preds[i].item() == 1:
#                 found_cases += 1
                
#                 # 1. 提取原始 3D MRI 图像
#                 img_3d = images[i, 0].cpu().numpy()  
                
#                 # 2. 提取注意力掩码，并插值放大到原始图像大小 (96, 96, 96)
#                 attn_tensor = spatial_attention[i:i+1] 
#                 attn_resized = F.interpolate(attn_tensor, size=img_3d.shape, mode='trilinear', align_corners=False)
#                 attn_3d = attn_resized[0, 0].cpu().numpy()
                
#                 # 3. 智能寻找网络最关注的切片位置
#                 slice_x = np.argmax(np.sum(attn_3d, axis=(1, 2))) 
#                 slice_y = np.argmax(np.sum(attn_3d, axis=(0, 2))) 
#                 slice_z = np.argmax(np.sum(attn_3d, axis=(0, 1))) 
                
#                 # 4. 绘图
#                 fig, axes = plt.subplots(1, 3, figsize=(18, 6))
#                 # 成功识别的pMCI阳性病例  高亮区域为网络空间对齐中点关注区域
#                 fig.suptitle(f"True Positive pMCI Case #{found_cases}\n(Spatial Attention Guided Alignment)", fontsize=18, fontweight='bold', y=1.05)
                
#                 # 矢状面 矢状面切片
#                 axes[0].imshow(img_3d[slice_x, :, :], cmap='gray') 
#                 im = axes[0].imshow(attn_3d[slice_x, :, :], cmap='jet', alpha=0.45) 
#                 axes[0].set_title(f"Sagittal View (Slice {slice_x})", fontsize=14)
#                 axes[0].axis('off')
                
#                 # 冠状面 冠状面切片
#                 axes[1].imshow(img_3d[:, slice_y, :], cmap='gray')
#                 axes[1].imshow(attn_3d[:, slice_y, :], cmap='jet', alpha=0.45)
#                 axes[1].set_title(f"Coronal View (Slice {slice_y})", fontsize=14)
#                 axes[1].axis('off')
                
#                 # 轴状面 轴状面切片
#                 axes[2].imshow(img_3d[:, :, slice_z], cmap='gray')
#                 axes[2].imshow(attn_3d[:, :, slice_z], cmap='jet', alpha=0.45)
#                 axes[2].set_title(f"Axial View (Slice {slice_z})", fontsize=14)
#                 axes[2].axis('off')
                
#                 # 颜色条的标签 解剖引导强度
#                 cbar = fig.colorbar(im, ax=axes.ravel().tolist(), shrink=0.7)
#                 cbar.set_label('Attention Weight', fontsize=12)
                
#                 plt.show()
                
#                 if found_cases >= num_cases_to_show:
#                     print(f"\n✅ 成功提取 {num_cases_to_show} 组 pMCI 空间注意力图！")
#                     return

#     print(f"\n搜索结束，共找到 {found_cases} 个符合条件的病例。")

# # 执行探针！展示前 5 个最典型的病例
# visualize_pMCI_attention(vis_model, val_loader, DEVICE, num_cases_to_show=5)